In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, Dataset
import sys 
sys.path.append("../")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 10

# Load the STL-10 dataset
train_ds = torchvision.datasets.STL10(root='./data', split='train', download=True, transform=transforms.Resize((32, 32)))
unlabeled_base_ds = torchvision.datasets.STL10(root='./data', split='unlabeled', download=True, transform=transforms.Resize((32, 32)))
test_ds = torchvision.datasets.STL10(root='./data', split='test', download=True, transform=transforms.Resize((32, 32)))

print(f"Training samples: {len(train_ds)}, Unlabeled samples: {len(unlabeled_base_ds)}, Test samples: {len(test_ds)}")

mean = torch.tensor([0.4409, 0.4279, 0.3868])
std = torch.tensor([0.2309, 0.2262, 0.2237])

Training samples: 5000, Unlabeled samples: 100000, Test samples: 8000


In [7]:
class RandomRangAugment(transforms.RandAugment):
    def __init__(self, num_ops=2):
        super().__init__(num_ops=num_ops)

    def __call__(self, img):
        # just randaugment with random magnitude
        self.magnitude = torch.randint(0, self.num_magnitude_bins, (1,)).item()
        img = super().__call__(img)
        return img

In [8]:
norm_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

weak_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

strong_transform=transforms.Compose([
        RandomRangAugment(num_ops=2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
        transforms.RandomErasing(),
    ])

In [9]:
class UnlabeledDataset(Dataset):
    def __init__(self, dataset, weak_transform, strong_transform):
        self.dataset = dataset
        self.weak_transform = weak_transform
        self.strong_transform = strong_transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, _ = self.dataset[idx]
        x_w = self.weak_transform(x)
        x_s = self.strong_transform(x)
        return x_w, x_s, idx

In [10]:
from datasets import TransformedDataset
from wideresnet2 import WideResNet 
from utils import evaluate_f1_and_accuracy
import os
import json
import time
import numpy as np

num_runs = 3
alpha = 0.75
T = 0.5

test_ds = TransformedDataset(test_ds, norm_transform)

for run in range(num_runs):
    print(f"Run {run+1}/{num_runs}")

    # Init random seeds for reproducibility
    torch.manual_seed(run)
    num_labeled = 40
    perm_indices = torch.randperm(len(train_ds))
    labeled_indices = perm_indices[:num_labeled]
    unlabeled_indices = perm_indices[num_labeled:]

    # Create datasets and dataloaders
    labeled_ds = TransformedDataset(Subset(train_ds, labeled_indices), weak_transform)
    # conctanenate the unlabeled dataset with the unlabeled split of the original dataset to ensure we have the correct number of unlabeled samples
    unlabeled_concat_ds = torch.utils.data.ConcatDataset([
        Subset(train_ds, unlabeled_indices),
        unlabeled_base_ds
    ])
    unlabeled_ds = UnlabeledDataset(unlabeled_concat_ds, weak_transform, weak_transform)

    # Create dataloaders
    batch_size = 64
    mu = 1
    labeled_loader = DataLoader(labeled_ds, batch_size=batch_size, shuffle=True)
    unlabeled_loader = DataLoader(unlabeled_ds, batch_size=batch_size*mu, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    # Create iterators for the dataloaders
    labeled_iter = iter(labeled_loader)
    unlabeled_iter = iter(unlabeled_loader)

    # Define the model
    model = WideResNet(depth=28, widen_factor=2, num_classes=num_classes).to(device)
    max_steps = 10_940  # Equivalent to 100 epochs on the full dataset with batch size 64
    optimizer = torch.optim.SGD(model.parameters(), lr=0.03, momentum=0.9, weight_decay=5e-4, nesterov=True)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_steps)

    # Settings
    method_name = "MixMatch"
    name_of_experiment = f"stl10_{num_labeled}_labels_run_{run+1}"

    if os.path.exists(f"results/{name_of_experiment}/{method_name}.json"):
        print(f"Results for {name_of_experiment} already exist. Skipping training to avoid overwriting.")
        continue # this will skip the rest of the training loop and move to the next run

    # Metrics to track
    metrics = {
    "test_f1": [0.0],  # Start with 0% F1 before training
    "test_acc": [0.0],  # Start with 0% accuracy before training
    "budget": [0]
    }

    # Hyperparameters
    alpha = 0.75
    T = 0.5
    budget_per_iteration = 1 + 2 * mu  # 1 batch of labeled + 2 batches of unlabeled (weak + weak)
    test_budget_period = 700  # Evaluate on test set every 700 batches seen

    # Training loop
    current_budget = 0
    start_time = time.time()
    for step in range(max_steps):
        running_loss = 0.0
        running_loss_sup = 0.0
        running_loss_unsup = 0.0
        model.train()
        try:
            x_l, y_l = next(labeled_iter)
        except StopIteration:
            labeled_iter = iter(labeled_loader)
            x_l, y_l = next(labeled_iter)

        try:
            x_u_1, x_u_2, _ = next(unlabeled_iter)
        except StopIteration:
            unlabeled_iter = iter(unlabeled_loader)
            x_u_1, x_u_2, _ = next(unlabeled_iter)

        x_l, y_l = x_l.to(device), y_l.to(device)
        x_u_1, x_u_2 = x_u_1.to(device), x_u_2.to(device)

        # unsupervised
        with torch.no_grad():
            logits_u_1 = model(x_u_1)
            logits_u_2 = model(x_u_2)
            probs_u_1 = F.softmax(logits_u_1, dim=1)
            probs_u_2 = F.softmax(logits_u_2, dim=1)
            probs_avg = (probs_u_1 + probs_u_2) / 2
            
            # Sharpening
            probs_sharpened = probs_avg ** (1 / T)
            probs_sharpened = probs_sharpened / probs_sharpened.sum(dim=1, keepdim=True)

        # Concatenate labeled and unlabeled data
        all_inputs = torch.cat([x_l, x_u_1, x_u_2], dim=0)
        all_targets = torch.cat([F.one_hot(y_l, num_classes), probs_sharpened, probs_sharpened], dim=0)

        # Shuffle the combined data
        indices = torch.randperm(all_inputs.size(0))
        all_inputs = all_inputs[indices]
        all_targets = all_targets[indices]

        # Mixup
        lam_x = torch.tensor([np.random.beta(alpha, alpha) for _ in range(x_l.size(0))], dtype=torch.float32, device=device).unsqueeze(1).unsqueeze(2).unsqueeze(3)
        lam_u = torch.tensor([np.random.beta(alpha, alpha) for _ in range(x_u_1.size(0) + x_u_2.size(0))], dtype=torch.float32, device=device).unsqueeze(1).unsqueeze(2).unsqueeze(3)

        mixup_x = lam_x.view(-1, 1, 1, 1) * x_l + (1 - lam_x.view(-1, 1, 1, 1)) * all_inputs[:x_l.size(0)]
        mixup_u = lam_u.view(-1, 1, 1, 1) * torch.cat([x_u_1, x_u_2], dim=0) + (1 - lam_u.view(-1, 1, 1, 1)) * all_inputs[x_l.size(0):]

        mixup_targets_x = lam_x.view(-1, 1) * F.one_hot(y_l, num_classes) + (1 - lam_x.view(-1, 1)) * all_targets[:x_l.size(0)]
        mixup_targets_u = lam_u.view(-1, 1) * torch.cat([probs_sharpened, probs_sharpened], dim=0) + (1 - lam_u.view(-1, 1)) * all_targets[x_l.size(0):]

        # Forward pass
        logits_mixup_x = model(mixup_x)
        logits_mixup_u = model(mixup_u)

        # Compute losses
        loss_x = F.cross_entropy(logits_mixup_x, mixup_targets_x.argmax(dim=1))

        #Sharpen logits before computing MSE loss
        logits_mixup_u_sharpened = logits_mixup_u ** (1 / T)
        logits_mixup_u_sharpened = logits_mixup_u_sharpened / logits_mixup_u_sharpened.sum(dim=1, keepdim=True)

        loss_u = F.mse_loss(logits_mixup_u_sharpened, mixup_targets_u)
        loss= loss_x + (min(step, 16000)/16000) * 100 * loss_u

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        running_loss_sup += loss_x.item()
        running_loss_unsup += loss_u.item()

        current_budget += budget_per_iteration

        if current_budget % test_budget_period < budget_per_iteration:
            f1, accuracy = evaluate_f1_and_accuracy(model, test_loader, device)
            metrics["test_f1"].append(f1)
            metrics["test_acc"].append(accuracy)
            metrics["budget"].append(current_budget)
            elapsed_time = time.time() - start_time
            print(f"Step {step+1}/{max_steps}, Budget: {current_budget}, Loss: {running_loss:.4f}, Sup Loss: {running_loss_sup:.4f}, Unsup Loss: {running_loss_unsup:.4f}, Test F1: {f1:.4f}, Test Acc: {accuracy:.4f}, Elapsed Time: {elapsed_time:.2f}s", flush=True)

            os.makedirs("results", exist_ok=True)
            os.makedirs(f"results/{name_of_experiment}", exist_ok=True)
            with open(f"results/{name_of_experiment}/{method_name}.json", "w") as f:
                json.dump(metrics, f, indent=4)

            # Early stopping if divergence is detected (accuracy does not go above 11% for 5 consecutive evaluations)
            early_stopping = (num_labeled == 40) and len(metrics["test_acc"]) > 5 and all(acc <= 0.11 for acc in metrics["test_acc"][-6:-1])
            if early_stopping:
                print(f"Early stopping at step {step+1} due to accuracy stagnating.")
                
                # Fill the remaining metrics with the last known values until max_steps
                for remaining_step in range(step+1, max_steps):
                    current_budget += budget_per_iteration
                    if current_budget % test_budget_period < budget_per_iteration:
                        metrics["test_f1"].append(metrics["test_f1"][-1])  # Append last known F1
                        metrics["test_acc"].append(metrics["test_acc"][-1])  # Append last known accuracy
                        metrics["budget"].append(current_budget)
                # Save the final metrics after early stopping
                with open(f"results/{name_of_experiment}/{method_name}.json", "w") as f:
                    json.dump(metrics, f, indent=4)
                break  

        elif step+1 == max_steps: # Final evaluation at the end of training
            f1, accuracy = evaluate_f1_and_accuracy(model, test_loader, device)
            metrics["test_f1"].append(f1)
            metrics["test_acc"].append(accuracy)
            metrics["budget"].append(current_budget)
            elapsed_time = time.time() - start_time
            print(f"Step {step+1}/{max_steps}, Budget: {current_budget}, Loss: {running_loss:.4f}, Sup Loss: {running_loss_sup:.4f}, Unsup Loss: {running_loss_unsup:.4f}, Test F1: {f1:.4f}, Test Acc: {accuracy:.4f}, Elapsed Time: {elapsed_time:.2f}s", flush=True)

            os.makedirs("results", exist_ok=True)
            os.makedirs(f"results/{name_of_experiment}", exist_ok=True)
            with open(f"results/{name_of_experiment}/{method_name}.json", "w") as f:
                json.dump(metrics, f, indent=4) 

        else:
            print(f"Step {step+1}/{max_steps}, Budget: {current_budget}, Loss: {running_loss:.4f}, Sup Loss: {running_loss_sup:.4f}, Unsup Loss: {running_loss_unsup:.4f}", end="\r", flush=True)

Run 1/3
Step 234/10940, Budget: 702, Loss: 1.0068, Sup Loss: 0.9792, Unsup Loss: 0.0189, Test F1: 0.2023, Test Acc: 0.2482, Elapsed Time: 30.73s
Step 467/10940, Budget: 1401, Loss: 0.5635, Sup Loss: 0.5121, Unsup Loss: 0.0177, Test F1: 0.2417, Test Acc: 0.2826, Elapsed Time: 61.25s
Step 700/10940, Budget: 2100, Loss: 0.7173, Sup Loss: 0.6477, Unsup Loss: 0.0159, Test F1: 0.2099, Test Acc: 0.2494, Elapsed Time: 92.02s
Step 934/10940, Budget: 2802, Loss: 0.4828, Sup Loss: 0.3884, Unsup Loss: 0.0162, Test F1: 0.2690, Test Acc: 0.3103, Elapsed Time: 122.79s
Step 1167/10940, Budget: 3501, Loss: 0.7056, Sup Loss: 0.5848, Unsup Loss: 0.0166, Test F1: 0.2351, Test Acc: 0.2711, Elapsed Time: 153.66s
Step 1400/10940, Budget: 4200, Loss: 0.5824, Sup Loss: 0.4540, Unsup Loss: 0.0147, Test F1: 0.2358, Test Acc: 0.2796, Elapsed Time: 184.67s
Step 1634/10940, Budget: 4902, Loss: 0.4779, Sup Loss: 0.3244, Unsup Loss: 0.0150, Test F1: 0.2261, Test Acc: 0.2690, Elapsed Time: 214.50s
Step 1867/10940, Bud